**[🏠 Course Home](00_START_HERE.ipynb)** | **🤖 [AI-Native](../ai/11_t_test_frequentist_vs_bayesian.ipynb)** | **🐍 [Python](../python/11_t_test_frequentist_vs_bayesian.ipynb)** | **🔵 [R](../r/11_t_test_frequentist_vs_bayesian.ipynb)** | **⬅️ [Prev: Ch 10: Production HMC & Diagnostics](10_mcmc_production_diagnostics.ipynb)** | **➡️ [Next: Ch 12: Bayesian Regression, Counts & GLMs](12_bayesian_regression_and_glms.ipynb)**

*🧭 Part III: Applied Modeling, Regression, Hierarchy & Model Comparison — Chapter 11 of 23*

---

# Chapter 11: Comparing Two Groups — The $t$-Test
### *From Null Hypothesis Significance Testing (NHST) to Bayes Factors & Bayesian Estimation (BEST)*

---

## 🧭 Executive Summary: The A/B Testing Dilemma

Every software engineer and product team runs two-sample comparisons:
* *Variant A (Baseline Control)* vs. *Variant B (New Pipeline / Redesigned UI)*.
* Did Variant B reduce latency? Did it improve conversion? Or was the observed gain just a fluke of random noise?

For 90 years, the default tool for this job has been **Null Hypothesis Significance Testing (NHST)** and Student's/Welch's $t$-test. But in production engineering, classical NHST suffers from three catastrophic conceptual failure modes.

---

## 1. The Three Fatal Fallacies of NHST & The $p$-Value

### Fallacy 1: The Large-$N$ Trap (Statistical vs. Practical Significance)
In modern web applications, experiments easily collect $N = 500{,}000$ requests.
* Because the $t$-statistic is proportional to $\sqrt{N}$, **any difference—no matter how microscopic—will eventually yield $p < 0.001$**.
* If Variant B is faster by $0.0002\text{ ms}$ (a difference completely imperceptible to users and dwarfed by network jitter), a frequentist $t$-test will declare:
  > *"Statistically significant improvement! Reject the null hypothesis!"*
* Teams end up celebrating and deploying changes that have zero real-world value.

### Fallacy 2: The Illusion of Posterior Probability
Engineers almost universally misinterpret $p = 0.03$ as meaning:
> ❌ *"There is only a 3% chance that the null hypothesis is true (or a 97% chance Variant B is better)."*
* **The Reality:** A $p$-value is $P(\text{Data as extreme as observed} \mid H_0)$. It is a statement about hypothetical data under an assumed null universe. It tells you **nothing** about $P(H_0 \mid \text{Data})$.

### Fallacy 3: Absence of Evidence $\neq$ Evidence of Absence
When $p = 0.18$, what can the frequentist say?
* Only that we *"fail to reject the null hypothesis."*
* We cannot say Variant A and Variant B are equal! The test might simply be underpowered. Frequentist NHST cannot accept the null hypothesis.

---

## 2. The Bayesian Revolution: Two Complementary Routes

To escape these traps, Bayesian statistics offers two clean, principled routes:

```
                               Two-Sample Comparison
                                        |
                 +----------------------+----------------------+
                 |                                             |
       Route 1: Model Comparison                     Route 2: Parameter Estimation
         (Bayes Factor BF_10)                             (Kruschke's BEST)
                 |                                             |
   "Which hypothesis is better supported:          "What is the actual magnitude of the
    the difference is zero (M_0) or non-zero (M_1)?" difference, its uncertainty, and its ROPE?"
```

---

## 3. Route 1: The Courtroom Scale (JZS Bayes Factor)

When executives demand a binary decision (*"Should we ship this feature or kill it?"*), model comparison uses the **Bayes Factor ($BF_{10}$)**:

$$BF_{10} = \frac{P(y \mid \mathcal{M}_1)}{P(y \mid \mathcal{M}_0)}$$

* $BF_{10} = 1$: Data is equally likely under both models (neutral).
* $BF_{10} = 10$: The data are $10\times$ more likely under the alternative model $\mathcal{M}_1$ than under the null $\mathcal{M}_0$.
* $BF_{10} = 0.1$: The data are $10\times$ more likely under the **null** model ($BF_{01} = 10$). **Bayes factors can provide direct positive evidence for the null!**

### The Jeffreys-Zellner-Siow (JZS) Prior
How do we define the alternative model without artificially gaming the test?
* The **JZS prior** places a Cauchy distribution with scale $r = \frac{\sqrt{2}}{2} \approx 0.707$ on the standardized effect size $\delta$.
* It is scale-invariant, symmetric, and acts as an impartial benchmark endorsed across scientific disciplines.

---

## 4. Route 2: Kruschke's BEST & The ROPE Decision Framework

In engineering, the null hypothesis is usually pointless:
> *"Is the latency difference between Version A and Version B exactly 0.00000000 ms?"*  
> Almost certainly not. Every code commit touches instruction caches and memory layouts.

Instead of testing whether a difference is zero, **Bayesian Estimation Supersedes the $t$-Test (BEST)** models the two groups directly:
1. It estimates $\mu_A, \mu_B$ and independent standard deviations $\sigma_A, \sigma_B$.
2. It yields the complete posterior distribution of difference $\Delta\mu = \mu_B - \mu_A$.

### The ROPE Ruler (Region of Practical Equivalence)
A **ROPE** defines the zone of operational indifference (e.g. any latency difference within $[-2.0\text{ ms}, +2.0\text{ ms}]$ is too small to matter).

We make decisions by overlaying the **95% Highest Density Interval (HDI)** onto the **ROPE**:

```
 Case 1: Meaningful Improvement (SHIP IT)
               [--- 95% HDI ---]
 --------------------------------|=== ROPE ===|------------------> Δμ
                               -2             +2

 Case 2: Proven Equivalence (NO MEANINGFUL DIFFERENCE)
                                  [ 95% HDI ]
 --------------------------------|=== ROPE ===|------------------> Δμ
                               -2             +2

 Case 3: Inconclusive (COLLECT MORE DATA)
                         [------ 95% HDI ------]
 --------------------------------|=== ROPE ===|------------------> Δμ
                               -2             +2
```

1. **Meaningful Win:** 95% HDI lies entirely outside the ROPE to the left ($P(\Delta\mu < -2\text{ ms}) > 95\%$).
2. **True Practical Equivalence:** 95% HDI lies entirely inside the ROPE ($P(|\Delta\mu| \le 2\text{ ms}) > 95\%$).
3. **Inconclusive:** 95% HDI overlaps the ROPE boundary. Continue testing.

### Explicit 3-Rule ROPE Operational Decision Matrix

To eliminate guesswork from production releases, the relationship between the $95\%$ Highest Density Interval (HDI) and the Region of Practical Equivalence (ROPE) maps directly to three operational engineering actions:

| Condition | Geometric HDI vs. ROPE Relationship | Statistical Meaning | Operational Engineering Decision |
| :--- | :--- | :--- | :--- |
| **Rule 1: Practical Superiority** | $95\%$ HDI lies **entirely outside** the ROPE in the favorable direction ($\text{HDI}_{\text{upper}} < -\delta_{\text{ROPE}}$). | Posterior probability of meaningful performance improvement exceeds $95\%$. Negligible effect ruled out. | **DEPLOY / SHIP IT**: The candidate build delivers a verified improvement exceeding the operational threshold. |
| **Rule 2: Practical Equivalence** | $95\%$ HDI lies **entirely inside** the ROPE ($[-\delta_{\text{ROPE}}, +\delta_{\text{ROPE}}]$). | With $95\%$ probability, the true difference is smaller than the operational threshold of indifference. | **CERTIFY EQUIVALENCE**: Refactor or optimization has no negative performance impact; safe to adopt for cleanups or cost wins. |
| **Rule 3: Inconclusive** | $95\%$ HDI **overlaps** the ROPE boundary (spans both inside and outside the threshold). | Telemetry is insufficient to rule out either a negligible effect or a meaningful improvement. | **GATHER MORE DATA / HOLD DEPLOYMENT**: Sample size is too small; keep test running or retain baseline build. |

---

### Complete Worked A/B Test Numerical Walkthrough: Welch's $t$-Test vs. Bayesian BEST + ROPE

To see how frequentist hypothesis testing and Bayesian estimation diverge on identical telemetry, consider an actual RPC latency benchmark comparing two distributed service builds:

* **Current Production Build A**: $n_A = 100$ requests, sample mean $\bar{y}_A = 150.0\text{ ms}$, sample standard deviation $s_A = 20.0\text{ ms}$.
* **Candidate Build B**: $n_B = 100$ requests, sample mean $\bar{y}_B = 143.5\text{ ms}$, sample standard deviation $s_B = 22.0\text{ ms}$.

#### 1. Frequentist Welch's $t$-Test Calculation

Under Welch's two-sample $t$-test (which allows for unequal variances $\sigma_A^2 \ne \sigma_B^2$):

1. **Observed Difference in Sample Means**:

$$
\hat{\Delta} = \bar{y}_B - \bar{y}_A = 143.5\text{ ms} - 150.0\text{ ms} = -6.5\text{ ms}
$$

2. **Standard Error of the Difference ($\text{SE}_\Delta$)**:

$$
\text{SE}_\Delta = \sqrt{\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}} = \sqrt{\frac{20.0^2}{100} + \frac{22.0^2}{100}} = \sqrt{\frac{400.0}{100} + \frac{484.0}{100}} = \sqrt{4.0 + 4.84} = \sqrt{8.84} \approx 2.973\text{ ms}
$$

3. **Compute the $t$-Statistic**:

$$
t = \frac{\hat{\Delta} - 0}{\text{SE}_\Delta} = \frac{-6.5}{2.973} \approx -2.186
$$

4. **Degrees of Freedom & $p$-Value**:
   Using the Welch-Satterthwaite approximation, $\nu \approx 196.3$ degrees of freedom:

$$
p\text{-value} = 2 \times P(T_{\nu} \le -2.186) \approx 0.030
$$

   Because $p = 0.030 < 0.05$, standard automated CI tooling reports:  
   **"Statistically Significant Latency Reduction ($p = 0.03$) — SHIP IT!"**

#### 2. Bayesian Estimation + ROPE Decision Rule

Now evaluate the identical telemetry through Bayesian Estimation (BEST) with a Region of Practical Equivalence.

Suppose site reliability engineering sets the **ROPE** as:

$$
\text{ROPE} = [-5.0\text{ ms}, +5.0\text{ ms}]
$$

*Operational Justification*: A latency delta smaller than $5.0\text{ ms}$ is undetectable by end users and does not justify the operational risk of a fleet-wide canary deployment.

1. **Posterior Distribution of Difference $\Delta = \mu_B - \mu_A$**:
   Under weakly informative priors, the posterior distribution over the true mean difference is approximately Gaussian:

$$
\Delta \mid \mathcal{D} \sim \mathcal{N}\left(-6.5\text{ ms}, \; (2.973\text{ ms})^2\right)
$$

2. **$95\%$ Highest Density Interval (HDI)**:

$$
95\%\text{ HDI} = -6.5 \pm 1.96 \times 2.973 = -6.5 \pm 5.827 = [-12.33\text{ ms}, \; -0.67\text{ ms}]
$$

3. **Compute Posterior Probabilities Relative to the ROPE**:
   What is the probability that Candidate Build B yields a *meaningful* improvement exceeding the $5\text{ ms}$ threshold?
   Using the standard normal cumulative distribution function $\Phi(z)$:

$$
z = \frac{-5.0 - (-6.5)}{2.973} = \frac{1.5}{2.973} \approx 0.5045
$$

$$
P(\Delta < -5.0\text{ ms}) = \Phi(0.5045) \approx 69.3\%
$$

   This immediately reveals the hidden operational vulnerability:

$$
P(\Delta \in [-5.0\text{ ms}, +5.0\text{ ms}]) = 1 - 0.693 = 30.7\%
$$

4. **Applying the 3-Rule Decision Matrix**:
   - The $95\%$ HDI $[-12.33\text{ ms}, -0.67\text{ ms}]$ straddles the $-5.0\text{ ms}$ ROPE threshold.
   - It covers both meaningful wins (down to $-12.33\text{ ms}$) and practically trivial differences (up to $-0.67\text{ ms}$).
   - **Conclusion: Rule 3 Applies (Inconclusive / Gather More Data)**.  
     While the frequentist test claimed victory ($p = 0.03$), the Bayesian analysis proves there is a **$30.7\%$ probability that the true latency reduction is inside the negligible ROPE**. The correct engineering action is to hold deployment and collect an additional $100$ requests per group to shrink the posterior interval before risking a canary rollout.

---

## 5. The Peeking / Optional Stopping Trap

In fast-paced tech companies, teams check dashboards continuously:
* If $p < 0.05$ on Tuesday, they stop the test and declare victory.
* If not, they keep running until Friday.

### The Disaster of Frequentist Peeking:
* In frequentist statistics, stopping rules are baked into the mathematics of the $p$-value.
* Repeatedly peeking and stopping at the first sign of $p < 0.05$ inflates the false positive rate from $5\%$ to **$30\% - 60\%$**!

### Why Bayesian Updating is Immune:
* In Bayesian inference, evidence is evaluated condition-by-condition:
  $$p(\theta \mid y_1, y_2) \propto p(y_2 \mid \theta) \, p(\theta \mid y_1)$$
* You can inspect the posterior after every single sample. If you stop because the 95% HDI has shrunken entirely inside the ROPE, your conclusion remains mathematically valid because your posterior represents your updated state of belief given all data observed so far.

---

## 🏁 Summary Checklist: Which Route Should You Choose?

| Scenario | Recommended Approach | Reason |
| :--- | :--- | :--- |
| **Strict regulatory / journal publication** | Welch's $t$-test (Frequentist) | Fixed pre-registered sample size; standardized peer-review expectation. |
| **Direct evidence for equality ($A = B$)** | JZS Bayes Factor ($BF_{01}$) | Quantifies exact evidence supporting the null over the alternative. |
| **Production A/B testing & decision making** | Kruschke's BEST + ROPE | Prevents the large-$N$ trap; models effect sizes; enables continuous monitoring. |

---

* 🐍 **Explore the Runnable Python Lab**: [Sheet: Two-Sample Comparison (t-Test)](../python/11_t_test_frequentist_vs_bayesian.ipynb)
* 🔵 **Explore the Runnable R Lab**: [Sheet: Two-Sample Comparison (t-Test)](../r/11_t_test_frequentist_vs_bayesian.ipynb)

---
## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 10: Production HMC & Diagnostics](10_mcmc_production_diagnostics.ipynb)** *(Part II: Numerical & Sampling Engines)*
* ➡️ **Next**: **[Ch 12: Bayesian Regression, Counts & GLMs](12_bayesian_regression_and_glms.ipynb)** *(Part III: Applied Modeling, Regression, Hierarchy & Model Comparison)*
* 🔄 **Companion Tracks for Chapter 11**: [📖 Conceptual](../conceptual/11_t_test_frequentist_vs_bayesian.ipynb) | [🤖 AI-Native](../ai/11_t_test_frequentist_vs_bayesian.ipynb) | [🐍 Python](../python/11_t_test_frequentist_vs_bayesian.ipynb) | [🔵 R](../r/11_t_test_frequentist_vs_bayesian.ipynb)
